In [3]:
import numpy as np
import pandas as pd
from nltk.corpus import stopwords
from nltk.stem import SnowballStemmer

from crawler.config import BASE_DIR, MD_DIR

In [4]:
mappings = (
    pd.read_csv(BASE_DIR / "20240711165346.csv")
    .query("status == 'completed'")
    .set_index("doc_id")
)

In [5]:
stop_words = set(stopwords.words('english'))
stemmer = SnowballStemmer("english")

In [6]:
def preprocess(doc: str) -> list[str]:
    return [
        stemmer.stem(word)
        for word in doc.lower().split()
        if word.isascii() and word not in stop_words
    ]

In [7]:
def load_docs():
    for file in MD_DIR.glob('*_ENG.md'):
        yield file.stem, preprocess(file.read_text("utf-8"))

In [8]:
docs = [(name, doc) for name, doc in load_docs() if doc]

In [9]:
from rank_bm25 import BM25Okapi

In [10]:
bm25 = BM25Okapi([doc for _, doc in docs])

In [24]:
def retrieve_urls_for_best_docs(query: str, n: int = 10) -> list[str]:
    tokenized_query = preprocess(query)
    doc_scores = bm25.get_scores(tokenized_query)
    best_docs = np.argsort(doc_scores)[::-1]
    return [mappings.loc[docs[i][0].split("_")[0], "url"] for i in best_docs[:n]]

In [25]:
retrieve_urls_for_best_docs("covid-19")

['https://grants.tuebingen.mpg.de/6083/training-young-scientists',
 'https://ellis.eu/units/tubingen',
 'https://grants.tuebingen.mpg.de/14269/news',
 'https://portal.qbic.uni-tuebingen.de/portal/web/ncct/resources?p_p_id=49&p_p_lifecycle=1&p_p_state=normal&p_p_mode=view&_49_struts_action=%2Fmy_sites%2Fview&_49_groupId=33387&_49_privateLayout=false',
 'https://portal.qbic.uni-tuebingen.de/portal/web/ncct/bioinformatics-services?p_p_id=49&p_p_lifecycle=1&p_p_state=normal&p_p_mode=view&_49_struts_action=%2Fmy_sites%2Fview&_49_groupId=33387&_49_privateLayout=false',
 'https://portal.qbic.uni-tuebingen.de/portal/web/ncct/sequencing-solutions?p_p_id=49&p_p_lifecycle=1&p_p_state=normal&p_p_mode=view&_49_struts_action=%2Fmy_sites%2Fview&_49_groupId=33387&_49_privateLayout=false',
 'https://portal.qbic.uni-tuebingen.de/portal/web/ncct/ncct_welcome',
 'https://portal.qbic.uni-tuebingen.de/portal/web/ncct/submit-your-project?p_p_id=49&p_p_lifecycle=1&p_p_state=normal&p_p_mode=view&_49_struts_act